In [ ]:
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)


In [ ]:
# Import libraries here
import pandas as pd
from pymongo import MongoClient
import matplotlib.pyplot as plt
import plotly.express as px
from pprint import PrettyPrinter
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA


In [ ]:
host = "192.36.115.2"


In [ ]:
client = MongoClient(host=host, port=27017)
db = client["air-quality"]
dar = db["dar-es-salaam"]


In [ ]:
pp = PrettyPrinter(indent=2)
sites = dar.distinct("metadata.site")
sites


In [ ]:
result = dar.aggregate(
    [
        {"$group": {"_id":"$metadata.site","count":{"$count":{}}}}
    ]
)
readings_per_site = list(result)
readings_per_site


In [ ]:
def wrangle(collection):
    results = collection.find(
        {"metadata.site": 11, "metadata.measurement": "P2"},
        projection={"P2": 1, "timestamp": 1, "_id": 0},
    )
    df = pd.DataFrame(results).set_index("timestamp")

    # Localize timezone
    df.index = df.index.tz_localize("UTC").tz_convert("Africa/Dar_es_Salaam")

    # Remove outliers PM2.5 readings above 100
    df = df[df["P2"]<= 100]

    # Resample to 1H window & fill missing value
    y = df["P2"].resample("1H").mean().fillna(method="ffill")
  
    return y


In [ ]:
y = wrangle(dar)
print(type(y))


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
# use ax=ax in your plot
y.plot(kind="line", xlabel="Date", ylabel="PM2.5 Level", title="Dar es Salaam PM2.5 Levels", ax=ax)


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
# use ax=ax in your plot
y.rolling(168).mean().plot(ax=ax,xlabel="Date", ylabel= "PM2.5 Level", title="Dar es Salaam PM2.5 Levels, 7-Day Rolling Average")


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
# use ax=ax in your plot
plot_acf(y, ax=ax)
plt.xlabel("Lag [hours]")
plt.ylabel("Correlation Coefficient")
plt.title("Dar es Salaam PM2.5 Readings, ACF")


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
# Use ax=ax in your plot
plot_pacf(y, ax=ax)
plt.xlabel("Lag [hours]")
plt.ylabel("Correlation Coefficient")
plt.title("Dar es Salaam PM2.5 Readings, PACF")


In [ ]:
cutoff_test = int(len(y) * 0.90)
y_train = y.iloc[:cutoff_test]
y_test = y.iloc[cutoff_test:]
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


In [ ]:
y_train_mean = y_train.mean()
y_pred_baseline = [y_train_mean] * len(y_train)
mae_baseline = mean_absolute_error(y_train, y_pred_baseline)

print("Mean P2 Reading:", y_train_mean)
print("Baseline MAE:", mae_baseline)


In [ ]:
# Create range to test different lags
p_params = range(1, 31)

# Create empty list to hold mean absolute error scores
maes = []

# Iterate through all values of p in `p_params`
for p in p_params:
    # Build model
    model = AutoReg(y_train, lags=p).fit()

    # Make predictions on training data, dropping null values caused by lag
    y_pred = model.predict().dropna()

    # Calculate mean absolute error for training data vs predictions
    mae = mean_absolute_error(y_train.iloc[p:], y_pred)

    # Append `mae` to list `maes`
    maes.append(mae)

# Put list `maes` into Series with index `p_params`
mae_series = pd.Series(maes, name="mae", index=p_params)

# Inspect head of Series
mae_series.head(30)


In [ ]:
best_p = mae_series.idxmin()
best_model = AutoReg(y_train, lags=best_p).fit()


In [ ]:
y_train_resid = best_model.resid
y_train_resid.name = "residuals"
y_train_resid.head()


In [ ]:
# Plot histogram of residuals
fig, ax = plt.subplots()
y_train_resid.hist(ax=ax)
plt.xlabel("Residuals")
plt.ylabel("Frequency")
plt.title("Best Model, Training Residuals")


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
# Use ax=ax in your plot
plot_acf(y_train_resid,ax=ax)
plt.xlabel("Lag [hours]")
plt.ylabel("Correlation Coefficient")
plt.title("Dar es Salaam, Training Residuals ACF")


In [ ]:
y_pred_wfv = pd.Series(dtype=float)
history = y_train.copy()
for i in range(len(y_test)):
    model = AutoReg(history, lags=best_p).fit()
    next_pred = model.forecast(steps=1)
    
    # Store prediction
    y_pred_wfv = pd.concat([y_pred_wfv, next_pred])
    
    # Append actual value to history
    history = pd.concat([history, y_test.iloc[i:i+1]])
    
y_pred_wfv.name = "prediction"
y_pred_wfv.index.name = "timestamp"
y_pred_wfv.head()


In [ ]:
# Enter y_pred_wfv at ... (Ellipsis) to see the test mean absolute error

test_mae = mean_absolute_error(y_test, y_pred_wfv)
print("Test MAE (walk forward validation):", round(test_mae, 2))


In [ ]:
df_pred_test = pd.DataFrame({"y_test": y_test, "y_pred_wfv": y_pred_wfv})
fig = px.line(df_pred_test, labels={"Value": "PM2.5"})
fig.update_layout(
    title="Dar es Salaam, WFV Predictions",
    xaxis_title="Date",
    yaxis_title="PM2.5 Level",
)
